# EDA 3 — So sánh 2 bộ và gợi ý tiền xử lý
Đọc `reports/summary_*.json` do hai notebook trước sinh ra (chạy 01–02 trước).

In [1]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from eda_utils import *
warnings.filterwarnings('ignore'); pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
plt.rcParams['figure.dpi'] = 80
RAW = ROOT / 'data' / 'raw'
import json

In [2]:
S = {k: json.load(open(ROOT/'reports'/f'summary_{k}.json')) for k in ['bike','tetouan']}
rows = {
    'bike': {
        'license': 'CC BY 4.0', 'mục tiêu': 'cnt', 'ngoại sinh': 'thời tiết, lịch',
        'freq': S['bike']['freq'], 'rows': S['bike']['rows'], 'n_series': S['bike']['n_series'],
        'missing_steps': S['bike']['missing_steps'], 'n_na': S['bike']['n_na'],
        'seasonal_strength': S['bike'].get('seasonal_strength'), 'target_mean': S['bike'].get('target_mean'),
    },
    'tetouan': {
        'license': 'CC BY 4.0', 'mục tiêu': 'zone1, zone2, zone3', 'ngoại sinh': 'thời tiết, bức xạ',
        'freq': S['tetouan']['freq'], 'rows': S['tetouan']['rows'], 'n_series': S['tetouan']['n_series'],
        'missing_steps': S['tetouan']['missing_steps'], 'n_na': S['tetouan']['n_na'],
        'seasonal_strength': S['tetouan'].get('seasonal_strength'), 'target_mean': S['tetouan'].get('target_mean'),
    },
}
display(pd.DataFrame(rows).T)
for k in S: print(k, 'ACF:', S[k].get('acf'), '| dừng:', S[k].get('stationarity'))

,license,mục tiêu,ngoại sinh,freq,rows,n_series,missing_steps,n_na,seasonal_strength,target_mean
bike,CC BY 4.0,cnt,"thời tiết, lịch",giờ,17379,1,165,0,0.881,189.463
tetouan,CC BY 4.0,"zone1, zone2, zone3","thời tiết, bức xạ",10 phút,52416,3,0,0,0.987,32344.971


bike ACF: {'lag1': 0.843, 'lag2': 0.592, 'lag24': 0.819, 'lag168': 0.876} | dừng: {'cnt giờ': 'trái chiều: ADF nói dừng, KPSS nói không dừng (thường do xu hướng/mùa vụ → kiểm lại sau sai phân)', 'cnt ngày': 'không dừng', 'cnt ngày, sai phân 1': 'dừng'}
tetouan ACF: {'lag1': 0.922, 'lag3': 0.534, 'lag6': 0.005, 'lag24': 0.963, 'lag168': 0.953} | dừng: {'Zone 1 theo giờ': 'trái chiều: ADF nói dừng, KPSS nói không dừng (thường do xu hướng/mùa vụ → kiểm lại sau sai phân)'}


## Gợi ý tiền xử lý

- **Bike:** bỏ `casual`, `registered` và `instant`; giữ một trong `temp`/`atemp`; dựng lưới giờ đều; log1p nếu cần; tạo lag 24/168; chia theo thời gian.
- **Tetouan:** parse `DateTime`; chuẩn hoá tên ba target; scale từng target; tạo hour/day-of-week/month; quyết định giữ tần suất 10 phút hay resample theo giờ; dùng multi-output loss nếu dự báo ba zone đồng thời.
- **Chung:** fit scaler/imputer trên train; không random split; báo cáo naive và seasonal-naive; ghi rõ forecast horizon và quy tắc resample.
